# Task Values — dbutils.jobs.taskValues

> **DataMindAI | Ahmed**  
> Passing arbitrary values between tasks using `dbutils.jobs.taskValues`  
> **Student data pipeline:** Ingest → Transform → Dashboard

---
## How Task Values Work

| Step | API | Where |
|------|-----|-------|
| Publish a value | `dbutils.jobs.taskValues.set(key='x', value=10)` | Publisher notebook |
| Reference it | `{{tasks.my_task.values.x}}` | Job UI — task parameter field |
| Consume it | `dbutils.widgets.get('x')` | Consumer notebook |

The value flows: **notebook `.set()`** → **job task parameter `{{ }}`** → **notebook `.get()` via widget**

---
### Rules
- Keys must be **strings**
- Values must be **JSON-serialisable** (str, int, float, bool, list, dict)
- Max **48 KB** per value
- Values are **scoped to one run** — not persisted between runs
- Only Python notebooks can call `.set()` and `.get()`
- **Recommended:** use `{{ }}` parameter pattern to read, NOT `.get()` in notebooks

In [ ]:
# ── SETUP ────────────────────────────────────────────────────────────────
spark.sql('CREATE CATALOG IF NOT EXISTS demo_catalog')
spark.sql('CREATE SCHEMA IF NOT EXISTS demo_catalog.bronze')
spark.sql('CREATE SCHEMA IF NOT EXISTS demo_catalog.silver')
spark.sql('CREATE SCHEMA IF NOT EXISTS demo_catalog.gold')
print('Setup complete.')

---
## TASK 1 — ingest_students (PUBLISHES values)
---

## Task 1 — ingest_students

**Publishes three task values:**

| Key | Type | Value | Read by |
|-----|------|-------|---------|
| `record_count` | int | 10 | Task 2, Task 3 |
| `run_date` | string | '2024-01-15' | Task 2, Task 3 |
| `at_risk_raw` | int | 2 | Task 2 |

**Task 1 input parameter (from job-level dynamic ref):**
```
run_date  =  {{job.start_time.iso_date}}
```

In [ ]:
# ── ingest_students.py ──────────────────────────────────────────────────
# PUBLISHES: record_count, run_date, at_risk_raw

from pyspark.sql import Row
from pyspark.sql.functions import lit

try:
    run_date = dbutils.widgets.get('run_date')
except:
    run_date = '2024-01-15'

students = [
    Row(student_id='S001', name='Aisha Rahman',    course='Data Science', score=82, attendance=95),
    Row(student_id='S002', name='James Bradley',   course='Data Science', score=41, attendance=60),
    Row(student_id='S003', name='Priya Patel',     course='Engineering',  score=76, attendance=88),
    Row(student_id='S004', name='Carlos Mendez',   course='Engineering',  score=55, attendance=72),
    Row(student_id='S005', name='Sophie Williams', course='Mathematics',  score=91, attendance=98),
    Row(student_id='S006', name='Kwame Asante',    course='Mathematics',  score=38, attendance=55),
    Row(student_id='S007', name='Liu Wei',         course='Data Science', score=67, attendance=80),
    Row(student_id='S008', name='Emma Johnson',    course='Engineering',  score=74, attendance=85),
    Row(student_id='S009', name='Tariq Ahmed',     course='Data Science', score=29, attendance=40),
    Row(student_id='S010', name='Fatima Al-Sayed', course='Mathematics',  score=88, attendance=94),
]

df = (spark.createDataFrame(students)
        .withColumn('assessment_date', lit(run_date)))
df.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.bronze.student_assessments')

record_count = df.count()
at_risk_raw  = df.filter('score < 40 OR attendance < 50').count()

# ── PUBLISH task values ──────────────────────────────────────────────────
dbutils.jobs.taskValues.set(key = 'record_count', value = int(record_count))
dbutils.jobs.taskValues.set(key = 'run_date',     value = run_date)
dbutils.jobs.taskValues.set(key = 'at_risk_raw',  value = int(at_risk_raw))

print(f'Published: record_count={record_count}, run_date={run_date}, at_risk_raw={at_risk_raw}')
df.show()

---
## TASK 2 — transform_scores (CONSUMES from Task 1, PUBLISHES for Task 3)
---

## Task 2 — transform_scores

**Task 2 parameters in the Job UI (recommended {{ }} pattern):**
```
record_count  =  {{tasks.ingest_students.values.record_count}}
run_date      =  {{tasks.ingest_students.values.run_date}}
at_risk_raw   =  {{tasks.ingest_students.values.at_risk_raw}}
```

> The notebook receives these as plain strings via `dbutils.widgets.get()`.
> Always cast to the correct type: `int(dbutils.widgets.get('record_count'))`

In [ ]:
# ── transform_scores.py ─────────────────────────────────────────────────
# CONSUMES (via widget params): record_count, run_date, at_risk_raw
# PUBLISHES: pass_rate, at_risk_count, top_performer

from pyspark.sql.functions import when, col, avg as spark_avg

try:
    record_count = int(dbutils.widgets.get('record_count'))
    run_date     = dbutils.widgets.get('run_date')
    at_risk_raw  = int(dbutils.widgets.get('at_risk_raw'))
except:
    record_count, run_date, at_risk_raw = 10, '2024-01-15', 2

print(f'Received: record_count={record_count}, run_date={run_date}, at_risk_raw={at_risk_raw}')

if record_count == 0:
    dbutils.notebook.exit('SKIP: empty ingest')

df = spark.table('demo_catalog.bronze.student_assessments')

df_graded = df.withColumn('grade',
    when(col('score') >= 70, 'Distinction')
    .when(col('score') >= 55, 'Merit')
    .when(col('score') >= 40, 'Pass')
    .otherwise('Fail')
).withColumn('at_risk',
    when((col('score') < 40) | (col('attendance') < 50), True).otherwise(False)
)
df_graded.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.silver.student_grades')

total      = df_graded.count()
passed     = df_graded.filter("grade != 'Fail'").count()
at_risk_ct = df_graded.filter('at_risk = true').count()
avg_sc     = round(df_graded.agg(spark_avg('score')).collect()[0][0], 1)
pass_rate  = round((passed / total) * 100, 1)
top_row    = df_graded.orderBy(col('score').desc()).first()
top_str    = f"{top_row['name']} ({top_row['score']}% - {top_row['grade']})"

# ── Data quality check using at_risk_raw from Task 1 ────────────────────
if at_risk_ct > at_risk_raw:
    print(f'WARNING: at-risk grew {at_risk_raw} -> {at_risk_ct} after transform')

# ── PUBLISH task values for Task 3 ──────────────────────────────────────
dbutils.jobs.taskValues.set(key = 'pass_rate',     value = float(pass_rate))
dbutils.jobs.taskValues.set(key = 'at_risk_count', value = int(at_risk_ct))
dbutils.jobs.taskValues.set(key = 'top_performer', value = top_str)

print(f'Pass rate: {pass_rate}%  |  At-risk: {at_risk_ct}  |  Top: {top_str}')
df_graded.show()

---
## TASK 3 — dashboard_refresh (CONSUMES from Task 1 AND Task 2)
---

## Task 3 — dashboard_refresh

**Task 3 parameters (reads from Task 1 AND Task 2 directly):**
```
pass_rate      =  {{tasks.transform_scores.values.pass_rate}}
at_risk_count  =  {{tasks.transform_scores.values.at_risk_count}}
top_performer  =  {{tasks.transform_scores.values.top_performer}}
record_count   =  {{tasks.ingest_students.values.record_count}}
run_date       =  {{tasks.ingest_students.values.run_date}}
```

> Task 1's values (`record_count`, `run_date`) are available to Task 3 directly —
> Task 2 does not need to re-publish them.

In [ ]:
# ── dashboard_refresh.py ────────────────────────────────────────────────
try:
    pass_rate     = float(dbutils.widgets.get('pass_rate'))
    at_risk_count = int(dbutils.widgets.get('at_risk_count'))
    top_performer = dbutils.widgets.get('top_performer')
    record_count  = int(dbutils.widgets.get('record_count'))
    run_date      = dbutils.widgets.get('run_date')
except:
    pass_rate, at_risk_count = 70.0, 2
    top_performer = 'Sophie Williams (91% - Distinction)'
    record_count, run_date = 10, '2024-01-15'

status = 'ON TRACK' if pass_rate >= 70 else ('NEEDS ATTENTION' if pass_rate >= 50 else 'CRITICAL')

print('=' * 55)
print('   STUDENT ASSESSMENT DASHBOARD')
print(f'   Date: {run_date}   |   Students assessed: {record_count}')
print('=' * 55)
print(f'   Pass Rate      :  {pass_rate}%  --  {status}')
print(f'   At-Risk        :  {at_risk_count} student(s)')
print(f'   Top Performer  :  {top_performer}')

if at_risk_count > 0:
    print()
    print(f'   AT-RISK STUDENTS:')
    df = spark.table('demo_catalog.silver.student_grades')
    for r in df.filter('at_risk = true').orderBy('score').collect():
        print(f'   {r["name"]:22}  Score:{r["score"]}%  Att:{r["attendance"]}%  {r["grade"]}')

print('=' * 55)

from pyspark.sql import Row
spark.createDataFrame([Row(
    run_date=run_date, pass_rate=pass_rate,
    at_risk_count=at_risk_count, top_performer=top_performer
)]).write.format('delta').mode('append').saveAsTable('demo_catalog.gold.dashboard_history')
print('Summary written to gold.dashboard_history')

---
## Alternative Pattern — dbutils.jobs.taskValues.get()
---

## Alternative Pattern — `.get()` with `debugValue`

Use `.get()` with `debugValue` **only during local notebook development**.
The `debugValue` is a local fallback — it is **never used** when the notebook
runs as part of a job.

> **Do not use `default=` argument** — it silently swallows missing keys
> and makes production errors very hard to diagnose.

In [ ]:
# ── Development-safe pattern using debugValue ────────────────────────────
# Use this pattern when developing the notebook interactively.
# In production (running as a job task), debugValue is ignored.

record_count = dbutils.jobs.taskValues.get(
    taskKey    = 'ingest_students',
    key        = 'record_count',
    debugValue = 10              # only used in interactive mode
)

run_date = dbutils.jobs.taskValues.get(
    taskKey    = 'ingest_students',
    key        = 'run_date',
    debugValue = '2024-01-15'
)

print(f'record_count = {record_count}')
print(f'run_date     = {run_date}')

---
## Advanced — Passing a List via Task Values into For-Each
---

## Advanced Pattern — List as Task Value → For-Each

Publish a list of at-risk students as a task value from Task 2.
Use it as the input to a For-Each task.

**For-Each configuration:**
```
Inputs:       {{tasks.transform_scores.values.at_risk_list}}
Concurrency:  3
```

**Nested task parameters:**
```
student_id  =  {{input.student_id}}
name        =  {{input.name}}
score       =  {{input.score}}
```

In [ ]:
# ── Add this to the end of transform_scores.py ──────────────────────────
# Publishes a list of at-risk student dicts for the For-Each task
import json

at_risk_rows = (df_graded
    .filter('at_risk = true')
    .select('student_id', 'name', 'score', 'attendance', 'grade')
    .collect()
)

at_risk_list = [
    {'student_id': r['student_id'], 'name': r['name'],
     'score': r['score'], 'attendance': r['attendance'], 'grade': r['grade']}
    for r in at_risk_rows
]

print(f'At-risk list ({len(at_risk_list)} students):')
print(json.dumps(at_risk_list, indent=2))

dbutils.jobs.taskValues.set(key = 'at_risk_list', value = at_risk_list)
print('at_risk_list published.')

In [ ]:
# ── send_alert.py (nested For-Each task) ────────────────────────────────
# Runs once per entry in at_risk_list
import datetime

try:
    student_id = dbutils.widgets.get('student_id')
    name       = dbutils.widgets.get('name')
    score      = int(dbutils.widgets.get('score'))
except:
    student_id, name, score = 'S009', 'Tariq Ahmed', 29

print(f'Alert: {name} ({student_id}) — Score: {score}%')

from pyspark.sql import Row
spark.createDataFrame([Row(
    student_id=student_id, name=name, score=score,
    alerted_at=datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
)]).write.format('delta').mode('append').saveAsTable('demo_catalog.gold.student_alerts_log')
print(f'Alert logged for {name}.')